# Triton Model Repository and Client

The Week 3-4 deliverable is to deploy a model with Triton Inference Server. This notebook builds the mental model and local file structure you need before launching the server.

You will create a Triton model repository, write `config.pbtxt`, save a TorchScript model artifact, review the Docker command, and prepare a client request.

## What Triton does

Triton Inference Server is a model-serving runtime. It can host models from multiple frameworks and expose them over HTTP/gRPC.

Core capabilities:

- model repository loading,
- multiple model versions,
- HTTP and gRPC inference APIs,
- dynamic batching,
- concurrent model execution,
- metrics for Prometheus,
- support for frameworks such as PyTorch, ONNX Runtime, TensorRT, Python, and more.

## 0. Setup

This notebook can be run without Triton installed. The local artifact and repository creation cells are runnable with PyTorch only. The client cells are templates that run only when a Triton server is available.

In [ ]:
import json
import os
import shutil
import textwrap
from pathlib import Path

try:
    import torch
    import torch.nn as nn
except ImportError as exc:
    raise ImportError("PyTorch is required for artifact creation. Install it with: pip install torch") from exc

torch.manual_seed(42)
ROOT = Path.cwd()
REPO_DIR = Path("triton_model_repository")
MODEL_NAME = "tiny_fraud_scorer"
MODEL_DIR = REPO_DIR / MODEL_NAME
VERSION_DIR = MODEL_DIR / "1"
print(f"Repository will be created at: {REPO_DIR.resolve()}")

## 1. Triton model repository layout

A Triton repository is just a directory with a convention. For one PyTorch model, it usually looks like this:

```text
triton_model_repository/
└── tiny_fraud_scorer/
    ├── config.pbtxt
    └── 1/
        └── model.pt
```

The folder `1` is the model version. Triton can serve multiple versions if you add folders like `2`, `3`, and configure version policy.

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
VERSION_DIR.mkdir(parents=True, exist_ok=True)

print("Created repository directories:")
for path in sorted(REPO_DIR.rglob("*")):
    print(path)

## 2. Create a TorchScript model artifact

Triton can serve PyTorch through the PyTorch backend. One common artifact is a TorchScript `.pt` file.

The model below mirrors the small MLP from the latency notebook so the serving path stays focused.

In [ ]:
class TinyFraudScorer(nn.Module):
    def __init__(self, n_features=64, hidden=256, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, input__0):
        return self.net(input__0)

model = TinyFraudScorer().eval()
example_input = torch.randn(1, 64)
scripted = torch.jit.trace(model, example_input)
artifact_path = VERSION_DIR / "model.pt"
scripted.save(str(artifact_path))

print(f"Saved model artifact: {artifact_path}")
print(f"Artifact size: {artifact_path.stat().st_size / 1024:.1f} KB")

## 3. Write `config.pbtxt`

`config.pbtxt` tells Triton what backend to use, what inputs and outputs look like, and how batching should work.

For this model:

- input name: `input__0`
- input shape per example: `[64]`
- output name: `output__0`
- output shape per example: `[2]`
- max batch size: `128`

Triton prepends the batch dimension automatically when `max_batch_size > 0`.

In [ ]:
config_pbtxt = """
name: "tiny_fraud_scorer"
platform: "pytorch_libtorch"
max_batch_size: 128

input [
  {
    name: "input__0"
    data_type: TYPE_FP32
    dims: [64]
  }
]

output [
  {
    name: "output__0"
    data_type: TYPE_FP32
    dims: [2]
  }
]

dynamic_batching {
  preferred_batch_size: [8, 16, 32, 64]
  max_queue_delay_microseconds: 1000
}
""".strip() + "\n"

config_path = MODEL_DIR / "config.pbtxt"
config_path.write_text(config_pbtxt, encoding="utf-8")
print(config_path.read_text())

## 4. Check the repository tree

Before running Triton, always inspect the repo structure. Many deployment issues are just misplaced files or wrong version folder names.

In [ ]:
def print_tree(path, prefix=""):
    path = Path(path)
    entries = sorted(path.iterdir(), key=lambda item: (item.is_file(), item.name))
    for index, entry in enumerate(entries):
        connector = "└── " if index == len(entries) - 1 else "├── "
        print(prefix + connector + entry.name)
        if entry.is_dir():
            extension = "    " if index == len(entries) - 1 else "│   "
            print_tree(entry, prefix + extension)

print(REPO_DIR.name)
print_tree(REPO_DIR)

## 5. Run Triton with Docker

If Docker and an NVIDIA-compatible environment are available, run Triton from the terminal. The command below is shown as text so this notebook remains safe to run anywhere.

Use CPU-only Triton for learning if you do not have a GPU. For GPU, add the NVIDIA container runtime options your environment requires.

In [ ]:
repo_abs = REPO_DIR.resolve()
docker_command = f"""
docker run --rm \
  -p 8000:8000 \
  -p 8001:8001 \
  -p 8002:8002 \
  -v {repo_abs}:/models \
  nvcr.io/nvidia/tritonserver:24.10-py3 \
  tritonserver --model-repository=/models
""".strip()
print(docker_command)

## 6. Triton health and metadata endpoints

When Triton is running, these endpoints are useful first checks:

- `GET /v2/health/live`
- `GET /v2/health/ready`
- `GET /v2/models/tiny_fraud_scorer`
- `GET /metrics` on port `8002`

A service can be live but not ready if models are still loading or have failed to load.

In [ ]:
health_checks = [
    "curl http://localhost:8000/v2/health/live",
    "curl http://localhost:8000/v2/health/ready",
    "curl http://localhost:8000/v2/models/tiny_fraud_scorer",
    "curl http://localhost:8002/metrics | head",
]
print("Run these in a terminal after Triton starts:")
for command in health_checks:
    print(command)

## 7. HTTP client request shape

The raw Triton HTTP v2 inference endpoint is:

```text
POST /v2/models/{model_name}/infer
```

The JSON payload names the input tensor, shape, datatype, and data. This is not always the most efficient protocol, but it is easy to inspect while learning.

In [ ]:
sample_features = torch.randn(2, 64).tolist()
payload = {
    "inputs": [
        {
            "name": "input__0",
            "shape": [2, 64],
            "datatype": "FP32",
            "data": sample_features,
        }
    ],
    "outputs": [{"name": "output__0"}],
}
print(json.dumps(payload, indent=2)[:1200] + "\n...")

## 8. Optional: send a request if Triton is running

This cell uses `httpx`, which is already part of this project. It will skip gracefully if Triton is not running.

In [ ]:
import httpx

url = "http://localhost:8000/v2/models/tiny_fraud_scorer/infer"
try:
    response = httpx.post(url, json=payload, timeout=2.0)
    print(f"Status: {response.status_code}")
    print(response.text[:1000])
except Exception as error:
    print("Triton request skipped or failed. Start Triton with the Docker command above to run this cell.")
    print(f"Reason: {error}")

## 9. Dynamic batching intuition

Dynamic batching lets Triton wait briefly for multiple requests and combine them into a larger model batch.

Trade-off:

- larger batches usually improve throughput,
- waiting for a batch can increase latency,
- `max_queue_delay_microseconds` controls how long Triton may wait,
- preferred batch sizes help Triton choose efficient execution sizes.

For online APIs, dynamic batching is usually a tail-latency negotiation. You are buying throughput with a small amount of queueing delay.

In [ ]:
batching_notes = {
    "preferred_batch_size": [8, 16, 32, 64],
    "max_queue_delay_microseconds": 1000,
    "interpretation": "Wait up to 1 ms to form preferred batch sizes before executing.",
}
batching_notes

## 10. Simple load-test template

A load test sends many requests and records latency percentiles. This template is intentionally small. For heavier testing, use tools like Locust, k6, wrk, or Triton's `perf_analyzer`.

Run this only after Triton is serving the model.

In [ ]:
import concurrent.futures
import statistics
import time

def percentile(values, q):
    values = sorted(values)
    if not values:
        return float("nan")
    index = min(len(values) - 1, max(0, int(round((q / 100) * (len(values) - 1)))))
    return values[index]

def one_request():
    start = time.perf_counter()
    response = httpx.post(url, json=payload, timeout=5.0)
    elapsed_ms = (time.perf_counter() - start) * 1000
    return response.status_code, elapsed_ms

def run_load_test(total_requests=50, concurrency=5):
    latencies = []
    statuses = []
    start = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [executor.submit(one_request) for _ in range(total_requests)]
        for future in concurrent.futures.as_completed(futures):
            status, latency = future.result()
            statuses.append(status)
            latencies.append(latency)
    total_time = time.perf_counter() - start
    return {
        "requests": total_requests,
        "concurrency": concurrency,
        "successes": sum(status == 200 for status in statuses),
        "throughput_rps": total_requests / total_time,
        "mean_ms": statistics.mean(latencies),
        "p50_ms": percentile(latencies, 50),
        "p95_ms": percentile(latencies, 95),
        "p99_ms": percentile(latencies, 99),
    }

try:
    print(run_load_test(total_requests=20, concurrency=4))
except Exception as error:
    print("Load test skipped. Start Triton first, then rerun this cell.")
    print(f"Reason: {error}")

## 11. Triton Perf Analyzer

Triton ships with `perf_analyzer`, which is usually better than a handmade notebook load test. It can sweep concurrency and report latency/throughput.

Example command:

```bash
perf_analyzer -m tiny_fraud_scorer \
  -u localhost:8001 \
  -i grpc \
  --concurrency-range 1:16:2 \
  --shape input__0:64
```

Use this when you want a serious benchmark report.

## 12. What to monitor in production

At minimum, an inference service should expose or collect:

- request rate,
- error rate,
- p50/p95/p99 latency,
- queue time,
- model execution time,
- CPU/GPU utilization,
- memory usage,
- batch size distribution,
- model version,
- input/schema validation failures.

Triton exposes metrics on port `8002`, which Prometheus can scrape.

## 13. Exercises

1. Change `max_queue_delay_microseconds` from `1000` to `100` and reason about latency vs throughput.
2. Add a second model version folder named `2` and write down how you would test version rollout.
3. Change `preferred_batch_size` to `[4, 8, 16]` and run `perf_analyzer`.
4. Try converting the model to ONNX and serving it with Triton's ONNX Runtime backend.
5. Write a one-page serving report: model artifact, server command, benchmark table, bottlenecks, next optimizations.

## Summary

You created the core Triton deployment assets:

- a model repository,
- a versioned model folder,
- a TorchScript model artifact,
- a `config.pbtxt`,
- a Docker launch command,
- an HTTP request payload,
- and a small load-test template.

This is the skeleton of the Week 3-4 deliverable. The next level is running Triton locally and collecting real latency/throughput results.